# T12 · trigrama online en Colab

Carga el checkpoint causal existente, conecta el trigrama oficial y evalúa val completa. Ejecuta las celdas en orden. No entrena la GRU ni evalúa test.

La GRU conserva PyTorch 2.5.1; el decodificador de palabras usa otro entorno CPU con PyTorch 1.13.1. Primero se comprueba que recibir los mismos logits juntos o por bloques produce la misma frase.

La primera preparación descarga `languageModel.tar.gz` (14,1 GB comprimido) y compila el runtime. El grafo preparado y los resultados se guardan en Drive. Si Dryad rechaza la descarga, la celda 7 permite usar una copia de Drive.


In [2]:
#@title 1. Abrir en Colab y elegir el proyecto
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab
except ImportError as error:
    raise RuntimeError("Ejecuta este notebook en Colab.") from error

previous = globals().get("PROJECT")
COLAB_BASE = Path(globals().get("COLAB_BASE", Path.cwd().resolve()))
PROJECT = COLAB_BASE / "tic_proyecto_trigrama_colab"
candidates = ([Path(previous)] if previous is not None else []) + [
    COLAB_BASE / "tic_proyecto_online_colab", COLAB_BASE / "tic_proyecto_colab",
]
PREVIOUS_PROJECT = next((candidate for candidate in candidates
                         if candidate != PROJECT and (candidate / "data/speech_2023").is_dir()), None)
CHECK_REPORT = None
print("Proyecto:", PROJECT)


Proyecto: /content/tic_proyecto_trigrama_colab


## Copiar el código

El paquete contiene las fuentes de esta fase y sus hashes.


In [3]:
#@title 2. Copiar el código del proyecto (snapshot incluido)
import base64
import hashlib
import io
import json
from pathlib import PurePosixPath
from zipfile import ZipFile

SOURCE_SHA256 = '93ff5cbcd488540c0ba10aa002b6de9257e9f8b0a31d97540b04b1b8f2d79d9a'
SOURCE_B64 = (
    '...'
)

SOURCE_BYTES = base64.b64decode(SOURCE_B64)
if hashlib.sha256(SOURCE_BYTES).hexdigest() != SOURCE_SHA256:
    raise ValueError("El paquete de código no supera su SHA-256.")
PROJECT.mkdir(parents=True, exist_ok=True)
with ZipFile(io.BytesIO(SOURCE_BYTES)) as archive:
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        if relative.is_absolute() or ".." in relative.parts or member.is_dir():
            raise ValueError(f"Ruta inesperada en el paquete: {member.filename}")
        destination = PROJECT.joinpath(*relative.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
SOURCE_MANIFEST = json.loads((PROJECT / "source_snapshot.json").read_text())
for name, expected in SOURCE_MANIFEST["files"].items():
    if hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Código distinto del snapshot: {name}")
print("Código preparado:", len(SOURCE_MANIFEST["files"]), "archivos")
print("Commit base:", SOURCE_MANIFEST["git_commit_base"])
print("SHA-256 del paquete:", SOURCE_SHA256)


Código preparado: 62 archivos
Commit base: 3ad8ac389c8932263bcc63048bab655bee2ed849
SHA-256 del paquete: 93ff5cbcd488540c0ba10aa002b6de9257e9f8b0a31d97540b04b1b8f2d79d9a


## Instalar el entorno de la GRU


In [4]:
#@title 3. Instalar el entorno del proyecto
import shutil

subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv==0.9.4"], check=True)
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("No se encontró uv después de instalarlo.")
PYTHON = PROJECT / ".venv" / "bin" / "python"
if not PYTHON.is_file():
    subprocess.run([UV, "venv", "--python", "3.12.11", "--seed", str(PROJECT / ".venv")], check=True)
subprocess.run([UV, "pip", "install", "--python", str(PYTHON), "-r", "requirements.txt"], cwd=PROJECT, check=True)
subprocess.run([UV, "pip", "check", "--python", str(PYTHON)], cwd=PROJECT, check=True)

def ejecutar(argumentos: list[str]) -> None:
    """Muestra stdout/stderr en Colab y conserva el motivo real si falla."""
    from collections import deque
    import signal

    recent = deque(maxlen=25)
    process = subprocess.Popen([str(PYTHON), *argumentos], cwd=PROJECT,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            recent.append(line)
        returncode = process.wait()
    finally:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
        process.stdout.close()
    if returncode:
        detail = "".join(recent).strip() or "El proceso terminó sin producir salida."
        raise RuntimeError(f"Falló {argumentos[0]} (código {returncode}).\n{detail}")

print("Entorno aislado preparado. No hace falta reiniciar el kernel.")


Entorno aislado preparado. No hace falta reiniciar el kernel.


## Elegir pesos y configuración


In [5]:
#@title 4. Conectar Drive y seleccionar el checkpoint existente
CARPETA_DRIVE = "speech_bci" #@param {type:"string"}
CHECKPOINT_RELATIVO = "speech_online_baseline/jaovk23p/mejor_val.pt" #@param {type:"string"}
DISPOSITIVO = "cuda" #@param ["cpu", "cuda"]

from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import drive
from uuid import uuid4
import json

relative = Path(CARPETA_DRIVE.strip())
checkpoint_relative = Path(CHECKPOINT_RELATIVO.strip())
if (relative.is_absolute() or ".." in relative.parts or not CARPETA_DRIVE.strip()
        or not CHECKPOINT_RELATIVO.strip() or checkpoint_relative.is_absolute()
        or ".." in checkpoint_relative.parts):
    raise ValueError("Usa rutas relativas a Mi unidad y a checkpoints, respectivamente.")
CHECK_REPORT = None
DRIVE_MOUNT = COLAB_BASE / "drive"
drive.mount(str(DRIVE_MOUNT))
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / relative
CHECKPOINT_PATH = DRIVE_ROOT / "checkpoints" / checkpoint_relative
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(CHECKPOINT_PATH)
experiment_id = datetime.now(ZoneInfo("Europe/Madrid")).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
RESULTS = DRIVE_ROOT / "experimentos" / ("trigrama_t12_" + experiment_id)
RESULTS.mkdir(parents=True, exist_ok=False)
(RESULTS / "fuentes.zip").write_bytes(SOURCE_BYTES)
(RESULTS / "source_snapshot.json").write_text(json.dumps(SOURCE_MANIFEST, indent=2), encoding="utf-8")
CONFIG = "configs/entrenos_online/speech_colab.yaml"
EVALUATION_CONFIG = "configs/entrenos_online/evaluacion_trigrama_colab.yaml"
LANGUAGE_CONFIG = "configs/entrenos_online/trigrama_colab.yaml"
configure = """
from pathlib import Path
import json, sys, torch, yaml
from src.inference.checkpoints import read_speech_checkpoint
from src.data.speech_dataset import file_sha256
state = read_speech_checkpoint(Path(sys.argv[1]))
if state['format_version'] != 3:
    raise ValueError('Selecciona un checkpoint entrenado con preprocesamiento causal.')
if sys.argv[5] == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Selecciona una GPU en el entorno de Colab o elige cpu.')
cfg = state['config']
Path(sys.argv[2]).write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
evaluation = yaml.safe_load(Path('configs/entrenos_online/speech_online_evaluacion.yaml').read_text())
evaluation.update(split='val', dispositivo=sys.argv[5], permitir_tf32=False)
Path(sys.argv[3]).write_text(yaml.safe_dump(evaluation, sort_keys=False), encoding='utf-8')
language = yaml.safe_load(Path('configs/entrenos_online/speech_trigrama.yaml').read_text())
language['modelo'] = str(Path(sys.argv[6]) / 'modelos_lenguaje' / 'speech_trigram')
Path(sys.argv[4]).write_text(yaml.safe_dump(language, sort_keys=False), encoding='utf-8')
print(json.dumps({'config':cfg, 'language':language, 'evaluation':evaluation,
                  'checkpoint_sha256':file_sha256(Path(sys.argv[1]))}))
"""
settings = json.loads(subprocess.check_output([str(PYTHON), "-c", configure, str(CHECKPOINT_PATH), CONFIG,
                                             EVALUATION_CONFIG, LANGUAGE_CONFIG, DISPOSITIVO, str(DRIVE_ROOT)],
                                            cwd=PROJECT, text=True))
cfg = settings["config"]
language_cfg = settings["language"]
CHECKPOINT_SHA256 = settings["checkpoint_sha256"]
for name in (CONFIG, EVALUATION_CONFIG, LANGUAGE_CONFIG):
    shutil.copyfile(PROJECT / name, RESULTS / Path(name).name)
shutil.copyfile(PROJECT / cfg["protocolo"]["manifiesto"], RESULTS / "particion_t12_v1.json")
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = str(settings["evaluation"]["hilos_cpu"])
data_link = PROJECT / "data" / "speech_2023"
if PREVIOUS_PROJECT is not None and not data_link.exists():
    data_link.parent.mkdir(parents=True, exist_ok=True)
    data_link.symlink_to(PREVIOUS_PROJECT / "data" / "speech_2023", target_is_directory=True)
    print("Se reutilizan los datos de la sesión anterior.")
print("Checkpoint:", CHECKPOINT_PATH)
print("SHA-256:", CHECKPOINT_SHA256)
print("Modelo:", cfg["modelo"])
print("Semilla de entrenamiento:", cfg["experimento"]["seed"], "· Evaluación:", DISPOSITIVO)
print("Resultados:", RESULTS)


Mounted at /content/drive
Checkpoint: /content/drive/MyDrive/speech_bci/checkpoints/speech_online_baseline/jaovk23p/mejor_val.pt
SHA-256: 853fa9c706c86ca75bcdba74208a4749389d24ca69c7309d535eb2e8697a1ebe
Modelo: {'nombre': 'speech_gru', 'entrada': 256, 'capa_dia': 256, 'capas': 5, 'unidades': 512, 'bidireccional': False, 'kernel': 14, 'stride': 4, 'dropout_entrada': 0.2, 'dropout_gru': 0.4}
Semilla de entrenamiento: 0 · Evaluación: cuda
Resultados: /content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3


## Datos de validación

Reutiliza las señales preparadas si siguen en esta sesión. En una sesión nueva, prepara el mismo protocolo fijo.


In [6]:
#@title 5. Obtener y extraer competitionData
ORIGEN_DATOS = "Drive" #@param ["Drive", "Dryad"]
ARCHIVO_DATOS_DRIVE = "" #@param {type:"string"}


def buscar_archivos_drive(root: Path) -> tuple[list[Path], bool]:
    """Busca el tar de competitionData por nombre, sin leer su contenido."""
    import time

    matches = []
    started = time.monotonic()
    visited = 0
    incomplete = False
    errors = []
    for current, directories, filenames in os.walk(root, onerror=errors.append, followlinks=False):
        visited += 1
        folder = Path(current)
        if len(folder.relative_to(root).parts) >= 5:
            incomplete = incomplete or bool(directories)
            directories.clear()
        else:
            directories[:] = sorted(name for name in directories if not name.startswith('.'))
        for name in sorted(filenames):
            if name.casefold().startswith('competitiondata') and name.casefold().endswith(('.tar.gz', '.tgz')):
                matches.append(folder / name)
        if visited >= 200 or time.monotonic() - started >= 15:
            incomplete = True
            break
    return matches, incomplete or bool(errors)


manifest = json.loads((PROJECT / "docs/datasets/speech_2023/manifest.json").read_text())
expected = next(item for item in manifest["files"] if item["name"] == "competitionData.tar.gz")
destination = PROJECT / "data" / "speech_2023" / expected["name"]
destination.parent.mkdir(parents=True, exist_ok=True)
my_drive = DRIVE_MOUNT / "MyDrive"
print("Tamaño esperado:", expected["size_bytes"], "bytes")
print("Espacio libre en Colab:", round(shutil.disk_usage(PROJECT).free / (1 << 30), 1), "GiB")
if ORIGEN_DATOS not in ("Drive", "Dryad"):
    raise ValueError("Selecciona Drive o Dryad en ORIGEN_DATOS.")

local_valid = False
if destination.is_file():
    try:
        ejecutar(["scripts/get_data.py", "--dataset", "speech_2023", "--files", expected["name"], "--verify-only"])
        local_valid = True
        print("Se reutiliza la copia local verificada.")
    except RuntimeError:
        print("La copia local no es válida; solo se sustituirá por un archivo verificado.")

if not local_valid and ORIGEN_DATOS == "Drive":
    if not my_drive.is_dir():
        raise RuntimeError("Drive no está disponible. Ejecuta la celda 4 para montarlo antes de continuar.")
    entered = ARCHIVO_DATOS_DRIVE.strip().strip("\"'")
    if entered:
        selected = Path(entered)
        if ".." in selected.parts:
            raise ValueError("Indica la ruta del archivo sin componentes '..'.")
        if selected.is_absolute():
            if not selected.is_relative_to(DRIVE_MOUNT):
                raise ValueError(f"La ruta debe estar dentro del Drive montado en {DRIVE_MOUNT}.")
            source = selected
        elif selected.parts and selected.parts[0] in ("MyDrive", "Shareddrives"):
            source = DRIVE_MOUNT / selected
        else:
            source = my_drive / selected
        if not source.is_file():
            raise FileNotFoundError(
                f"Colab no ve un archivo en: {source}\n"
                "En el panel Archivos, localízalo dentro de drive y usa 'Copiar ruta'. "
                "Comprueba que Drive está montado con la cuenta donde subiste el archivo. "
                "No se ha intentado descargar de Dryad."
            )
    else:
        candidates = [DRIVE_ROOT / "datos" / expected["name"], DRIVE_ROOT / expected["name"], my_drive / expected["name"]]
        source = next((candidate for candidate in candidates if candidate.is_file()), None)
        if source is None:
            print("Buscando competitionData en Mi unidad (hasta 5 niveles y 200 carpetas)...", flush=True)
            matches, incomplete = buscar_archivos_drive(my_drive)
            if incomplete:
                print("Búsqueda limitada: puede haber otras carpetas. Puedes indicar la ruta exacta.")
            complete = [candidate for candidate in matches if candidate.stat().st_size == expected["size_bytes"]]
            choices = complete or matches
            if len(choices) == 1:
                source = choices[0]
            else:
                if choices:
                    detail = "Se han encontrado varios archivos:\n" + "\n".join(str(candidate) for candidate in choices)
                else:
                    detail = "No se encontró el archivo. Rutas iniciales comprobadas:\n" + "\n".join(str(candidate) for candidate in candidates)
                raise RuntimeError(
                    f"{detail}\n\nEn el panel Archivos de Colab, localiza el archivo dentro de drive, "
                    "pulsa 'Copiar ruta' y pégala en ARCHIVO_DATOS_DRIVE. "
                    "Comprueba la cuenta de Drive si no aparece en el panel. "
                    "Vuelve a ejecutar solo esta celda. No se ha intentado descargar de Dryad."
                )
    print("Archivo encontrado en Drive:", source, flush=True)
    size = source.stat().st_size
    if size != expected["size_bytes"]:
        raise ValueError(f"El archivo de Drive tiene {size} bytes; se esperaban {expected['size_bytes']}. Comprueba que terminó la subida y que es el archivo original del 1 de septiembre de 2023.")
    temporary = destination.with_name(destination.name + ".drive.part")
    print("Copiando al disco de Colab; después se comprobará el SHA-256...", flush=True)
    shutil.copyfile(source, temporary)
    verification = """
from pathlib import Path
import sys
from scripts.get_data import leer_manifiesto, verificar
source = next(item for item in leer_manifiesto('speech_2023')['files'] if item['name'] == 'competitionData.tar.gz')
verificar(Path(sys.argv[1]), source)
print('Copia de Drive: tamaño y SHA-256 oficiales verificados.')
"""
    ejecutar(["-c", verification, str(temporary)])
    temporary.replace(destination)
elif not local_valid:
    ejecutar(["scripts/get_data.py", "--dataset", "speech_2023", "--files", expected["name"]])

extract = """
from pathlib import Path
import json
import tarfile
import yaml
cfg = yaml.safe_load(Path('configs/entrenos_online/speech_colab.yaml').read_text())
manifest = json.loads(Path('docs/datasets/speech_2023/manifest.json').read_text())
digest = next(item['sha256'] for item in manifest['files'] if item['name'] == 'competitionData.tar.gz')
marker = Path('data/speech_2023/.extraccion_verificada')
expected = [Path(cfg['datos']['raiz']) / split / (session + '.mat') for split in ['train', 'test'] for session in cfg['datos']['sesiones']]
if not marker.exists() or marker.read_text().strip() != digest or not all(path.is_file() for path in expected):
    with tarfile.open('data/speech_2023/competitionData.tar.gz') as archive:
        archive.extractall('data/speech_2023', filter='data')
    marker.write_text(digest)
print('Extraídas las señales originales. Las 19 sesiones seleccionadas están disponibles.')
"""
ejecutar(["-c", extract])


Tamaño esperado: 3669638707 bytes
Espacio libre en Colab: 187.4 GiB
Archivo encontrado en Drive: /content/drive/MyDrive/speech_bci/datos/competitionData.tar.gz
Copiando al disco de Colab; después se comprobará el SHA-256...
Copia de Drive: tamaño y SHA-256 oficiales verificados.
Extraídas las señales originales. Las 19 sesiones seleccionadas están disponibles.


In [7]:
#@title 6. Preparar recursos, fonemas y señales

ejecutar(["scripts/get_phoneme_resources.py", "--config", CONFIG])
labels = PROJECT / cfg["etiquetas"]["salida"] / "manifest.json"
signals = PROJECT / cfg["datos"]["preparados"] / "manifest.json"
if not labels.is_file():
    ejecutar(["scripts/prepare_phoneme_labels.py", "--config", CONFIG])
if not signals.is_file():
    ejecutar(["scripts/prepare_speech_data.py", "--config", CONFIG])
verify = """
from pathlib import Path
import json
import yaml
from src.data.speech_dataset import SpeechDataset
from src.data.speech_protocol import read_speech_protocol
cfg = yaml.safe_load(Path('configs/entrenos_online/speech_colab.yaml').read_text())
protocol = read_speech_protocol(cfg)
train = SpeechDataset(cfg, 'train')
val = SpeechDataset(cfg, 'val')
if (len(train), len(val), len(train.sessions)) != (5180, 1460, 19):
    raise ValueError('Los datos no corresponden al protocolo T12 fijado.')
if protocol['totals']['test'] != 680:
    raise ValueError('El test oficial ha cambiado.')
print(json.dumps({'train': len(train), 'val': len(val), 'test_reservado': 680,
                  'sesiones': len(train.sessions), 'normalizacion': 'solo train',
                  'hashes': 'verificados', 'protocolo_sha256': cfg['protocolo']['sha256']}, indent=2))
"""
ejecutar(["-c", verify])
shutil.copyfile(signals, RESULTS / "manifiesto_preparacion.json")
shutil.copyfile(labels, RESULTS / "manifiesto_fonemas.json")


Verificado y extraído: corpora/cmudict.zip
Verificado y extraído: taggers/averaged_perceptron_tagger.zip
Verificado y extraído: taggers/averaged_perceptron_tagger_eng.zip
Recursos de fonemas preparados y verificados.
/content/tic_proyecto_trigrama_colab/.venv/lib/python3.12/site-packages/g2p_en/g2p.py:156: SyntaxWarning: invalid escape sequence '\-'
  text = re.sub("[^ a-z'.,?!\-]", "", text)
Etiquetas preparadas: t12.2022.04.28
Etiquetas preparadas: t12.2022.05.05
Etiquetas preparadas: t12.2022.05.17
Etiquetas preparadas: t12.2022.05.19
Etiquetas preparadas: t12.2022.05.24
Etiquetas preparadas: t12.2022.05.26
Etiquetas preparadas: t12.2022.06.02
Etiquetas preparadas: t12.2022.06.07
Etiquetas preparadas: t12.2022.06.14
Etiquetas preparadas: t12.2022.06.16
Etiquetas preparadas: t12.2022.06.21
Etiquetas preparadas: t12.2022.06.28
Etiquetas preparadas: t12.2022.07.05
Etiquetas preparadas: t12.2022.07.14
Etiquetas preparadas: t12.2022.07.21
Etiquetas preparadas: t12.2022.07.27
Etiquetas pr

PosixPath('/content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3/manifiesto_fonemas.json')

## Preparar el trigrama

La compilación tarda y se realiza en la máquina de Colab. El entorno de la GRU se conserva. El grafo ya preparado en Drive se verifica y se reutiliza.


In [8]:
#@title 7. Obtener el grafo y compilar el runtime CPU
ORIGEN_LM = "Drive" #@param ["Dryad", "Drive"]
ARCHIVO_LM_DRIVE = "" #@param {type:"string"}

if ORIGEN_LM not in ("Drive", "Dryad"):
    raise ValueError("Selecciona Drive o Dryad en ORIGEN_LM.")
model_root = Path(language_cfg["modelo"])
archive_name = "languageModel.tar.gz"
destination = PROJECT / "data" / "speech_2023" / archive_name
if not (model_root / "manifest.json").is_file() and not destination.is_file():
    if ORIGEN_LM == "Drive":
        entered = ARCHIVO_LM_DRIVE.strip().strip("\"'")
        if entered:
            selected = Path(entered)
            if ".." in selected.parts:
                raise ValueError("Indica una ruta sin '..'.")
            if selected.is_absolute():
                if not selected.is_relative_to(DRIVE_MOUNT):
                    raise ValueError("El archivo debe estar en el Drive montado.")
                source = selected
            elif selected.parts and selected.parts[0] in ("MyDrive", "Shareddrives"):
                source = DRIVE_MOUNT / selected
            else:
                source = DRIVE_MOUNT / "MyDrive" / selected
        else:
            source = next((candidate for candidate in (DRIVE_ROOT / "datos" / archive_name,
                           DRIVE_ROOT / archive_name, DRIVE_MOUNT / "MyDrive" / archive_name)
                           if candidate.is_file()), None)
        if source is None or not source.is_file():
            raise FileNotFoundError("Indica la ruta de languageModel.tar.gz en ARCHIVO_LM_DRIVE.")
        destination.parent.mkdir(parents=True, exist_ok=True)
        expected = next(item for item in json.loads((PROJECT / "docs/datasets/speech_2023/manifest.json").read_text())["files"]
                        if item["name"] == archive_name)
        if source.stat().st_size != expected["size_bytes"]:
            raise ValueError("El archivo de Drive no tiene el tamaño oficial.")
        if shutil.disk_usage(PROJECT).free < source.stat().st_size + 5 * (1 << 30):
            raise RuntimeError("Falta espacio para copiar el archivo al disco de Colab.")
        temporary = destination.with_name(archive_name + ".drive.part")
        print("Copiando el archivo de Drive al disco de Colab...", flush=True)
        shutil.copyfile(source, temporary)
        verify = "from scripts.get_data import leer_manifiesto, verificar; from pathlib import Path; import sys; verificar(Path(sys.argv[1]), next(x for x in leer_manifiesto('speech_2023')['files'] if x['name']=='languageModel.tar.gz'))"
        ejecutar(["-c", verify, str(temporary)])
        temporary.rename(destination)
    else:
        ejecutar(["scripts/get_data.py", "--dataset", "speech_2023", "--files", archive_name])
ejecutar(["scripts/prepare_speech_trigram.py", "--config", LANGUAGE_CONFIG, "--prepare-model", "--build-runtime"])
runtime_python = PROJECT / language_cfg["python_runtime"]
shutil.copyfile(runtime_python.parent.parent / "runtime_build.json", RESULTS / "runtime_build.json")
shutil.copyfile(model_root / "manifest.json", RESULTS / "modelo_lenguaje.json")
print("Trigrama y runtime preparados.")


Cloning into '/content/tic_proyecto_trigrama_colab/data/language_models/speechBCI'...
HEAD is now at 522a960 Update boost link
Using CPython 3.10.15
Creating virtual environment with seed packages at: data/language_models/runtime
 + packaging==26.3
 + pip==26.2.2
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source data/language_models/runtime/bin/activate
Using Python 3.10.15 environment at: data/language_models/runtime
Resolved 7 packages in 715ms
Prepared 7 packages in 3.55s
Uninstalled 3 packages in 16ms
Installed 7 packages in 167ms
 + cmake==3.27.9
 + numpy==1.24.4
 - pip==26.2.2
 + pip==24.3.1
 - setuptools==84.2.1
 + setuptools==68.2.2
 + torch==1.13.1+cpu
 + typing-extensions==4.15.0
 - wheel==0.48.0
 + wheel==0.41.3
running install
/content/tic_proyecto_trigrama_colab/data/language_models/runtime/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        **************************************

## Registro en W&B


In [9]:
#@title 8. Cargar la credencial de W&B sin mostrarla
import getpass

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.utils.env import cargar_clave_wandb

if not cargar_clave_wandb(PROJECT / ".env", DRIVE_ROOT / ".env"):
    from google.colab import userdata
    try:
        api_key = userdata.get("WANDB_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        api_key = None
    if not api_key or not api_key.strip():
        api_key = getpass.getpass("API key de W&B (entrada oculta): ")
    if not api_key or not api_key.strip():
        raise ValueError("Falta la credencial de W&B.")
    os.environ["WANDB_API_KEY"] = api_key.strip()
    del api_key
print("Credencial cargada. W&B mostrará el enlace del run al iniciar la evaluación.")


Credencial cargada. W&B mostrará el enlace del run al iniciar la evaluación.


## Comprobar antes de evaluar

Usa una frase de val y compara el WFST con los mismos logits agrupados o por bloques. No representa una métrica del corpus.


In [10]:
#@title 9. Comprobar el estado incremental del trigrama
def evaluar_trigrama(destino: Path, *, solo_comprobar: bool = False) -> None:
    """Lanza la evaluación y muestra el log del trabajador si falla."""
    argumentos = ["scripts/evaluate_speech_trigram.py", "--checkpoint", str(CHECKPOINT_PATH),
                  "--evaluation-config", EVALUATION_CONFIG, "--language-config", LANGUAGE_CONFIG,
                  "--split", "val", "--output", str(destino)]
    if solo_comprobar:
        argumentos.append("--check-streaming")
    try:
        ejecutar(argumentos)
    except RuntimeError:
        log = destino / "trigram_worker.log"
        if log.is_file():
            print("Últimas líneas del trabajador:")
            print("\n".join(log.read_text(errors="replace").splitlines()[-40:]))
        raise

CHECK_REPORT = None
CHECK_OUTPUT = RESULTS / ("comprobacion_" + uuid4().hex[:8])
evaluar_trigrama(CHECK_OUTPUT, solo_comprobar=True)
report = json.loads((CHECK_OUTPUT / "comprobacion_incremental.json").read_text())
if not report["verified"] or report["checkpoint_sha256"] != CHECKPOINT_SHA256:
    raise RuntimeError("La comprobación incremental no pasó con el checkpoint seleccionado.")
CHECK_REPORT = report
print(json.dumps(CHECK_REPORT["profiles"], indent=2))


Comprobación incremental correcta: /content/drive/MyDrive/speech_bci/experimentos/trigrama_t12_20261101_210724_101948c3/comprobacion_399ade10
{
  "5": {
    "matches": true,
    "complete": "the giant redwood city in the rising sun",
    "streamed": "the giant redwood city in the rising sun",
    "frames": 121
  }
}


In [11]:
#@title Diagnosticar la carga del trigrama sin entrenar ni recompilar
from pathlib import Path
import json
import os
import signal
import subprocess
import tempfile
import time
from uuid import uuid4

try:
    import google.colab
except ImportError as error:
    raise RuntimeError("Esta celda se ejecuta en la sesión de Colab del notebook 13.") from error

if not all(name in globals() for name in ("PROJECT", "RESULTS", "language_cfg")):
    raise RuntimeError("Se necesitan las variables de las celdas 1 a 4 del notebook 13.")

project = Path(PROJECT)
python_runtime = project / language_cfg["python_runtime"]
graph = Path(language_cfg["modelo"]) / "TLG.fst"
worker = project / "scripts/trigram_worker.py"

def memoria_disponible() -> int:
    """Consulta la RAM disponible del sistema, en bytes."""
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) * 1024
    raise RuntimeError("No se pudo consultar MemAvailable.")

def eventos_memoria() -> str:
    """Lee los contadores del cgroup si el entorno permite consultarlos."""
    try:
        return Path("/sys/fs/cgroup/memory.events").read_text()
    except OSError:
        return "no disponible"

report = {"graph_bytes": graph.stat().st_size, "memory_events_before": eventos_memoria(),
          "cgroup": Path("/proc/self/cgroup").read_text(), "peak_worker_rss_bytes": 0,
          "stopped_by_probe": None}
# Umbrales de esta comprobación; no cambian la búsqueda ni los parámetros del modelo.
max_rss = 8 * (1 << 30)
min_available = 2 * (1 << 30)
max_seconds = 90
environment = {**os.environ, "CUDA_VISIBLE_DEVICES": ""}
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    environment[name] = str(language_cfg["hilos_cpu"])
request = {"op": "init", "model": str(graph.parent), "options": language_cfg["busqueda"],
           "threads": language_cfg["hilos_cpu"]}

print(f"Grafo: {report['graph_bytes'] / (1 << 30):.2f} GiB en disco", flush=True)
print("Prueba del runtime existente; parada a 8 GiB de RSS, menos de 2 GiB disponibles o 90 s.", flush=True)
with tempfile.TemporaryDirectory(prefix="trigram_load_probe_") as folder:
    stdout_path, stderr_path = Path(folder) / "stdout.log", Path(folder) / "stderr.log"
    with stdout_path.open("w") as out, stderr_path.open("w") as err:
        process = subprocess.Popen([str(python_runtime), "-u", "-X", "faulthandler", str(worker)],
                                   cwd=project, stdin=subprocess.PIPE, stdout=out, stderr=err,
                                   text=True, env=environment)
        started = last_print = time.monotonic()
        try:
            try:
                process.stdin.write(json.dumps(request) + "\n")
                process.stdin.close()
            except BrokenPipeError:
                pass
            while process.poll() is None:
                rss = 0
                try:
                    for line in Path(f"/proc/{process.pid}/status").read_text().splitlines():
                        if line.startswith("VmRSS:"):
                            rss = int(line.split()[1]) * 1024
                except FileNotFoundError:
                    continue
                report["peak_worker_rss_bytes"] = max(report["peak_worker_rss_bytes"], rss)
                available = memoria_disponible()
                now = time.monotonic()
                if rss >= max_rss or available < min_available:
                    report["stopped_by_probe"] = "límite preventivo de memoria de esta prueba"
                elif now - started >= max_seconds:
                    report["stopped_by_probe"] = "límite de tiempo de esta prueba"
                if report["stopped_by_probe"]:
                    process.terminate()
                    break
                if now - last_print >= 5:
                    print(f"RSS trabajador: {rss / (1 << 30):.2f} GiB; disponible: {available / (1 << 30):.2f} GiB", flush=True)
                    last_print = now
                time.sleep(0.1)
        finally:
            if not process.stdin.closed:
                try:
                    process.stdin.close()
                except BrokenPipeError:
                    pass
            if process.poll() is None:
                process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        report["returncode"] = process.returncode
        report["elapsed_seconds"] = time.monotonic() - started
    report["stdout"] = stdout_path.read_text(errors="replace")[-8192:]
    report["stderr"] = stderr_path.read_text(errors="replace")[-8192:]

report["memory_events_after"] = eventos_memoria()
if report["returncode"] < 0:
    try:
        report["signal"] = signal.Signals(-report["returncode"]).name
    except ValueError:
        report["signal"] = str(-report["returncode"])
report_path = Path(RESULTS) / ("diagnostico_carga_" + uuid4().hex[:8] + ".json")
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("Diagnóstico guardado:", report_path)


Grafo: 26.50 GiB en disco
Prueba del runtime existente; parada a 8 GiB de RSS, menos de 2 GiB disponibles o 90 s.
RSS trabajador: 0.93 GiB; disponible: 80.50 GiB
RSS trabajador: 2.16 GiB; disponible: 79.26 GiB
RSS trabajador: 3.44 GiB; disponible: 78.00 GiB
RSS trabajador: 4.72 GiB; disponible: 76.72 GiB
RSS trabajador: 6.01 GiB; disponible: 75.41 GiB
RSS trabajador: 7.33 GiB; disponible: 74.08 GiB
{
  "graph_bytes": 28458450578,
  "memory_events_before": "low 0\nhigh 0\nmax 0\noom 0\noom_kill 0\noom_group_kill 0\n",
  "cgroup": "0::/../../jupyter-children\n",
  "peak_worker_rss_bytes": 8594751488,
  "stopped_by_probe": "límite preventivo de memoria de esta prueba",
  "returncode": -15,
  "elapsed_seconds": 33.24799556800008,
  "stdout": "{\"ok\": true, \"python\": \"3.10.15\", \"torch\": \"1.13.1+cpu\", \"numpy\": \"1.24.4\"}\n",
  "stderr": "WARNING: Logging before InitGoogleLogging() is written to STDERR\nI1101 20:32:26.044090 28974 brain_speech_decoder.h:52] Reading fst /content/dr

## Evaluar val completa

Registra WER, PER, parciales y tiempos. Las llegadas se simulan sin red; los parciales de palabras pueden revisarse con nuevos bloques. Test sigue reservado.


In [12]:
#@title 10. Evaluar el trigrama en val
if not globals().get("CHECK_REPORT") or not CHECK_REPORT.get("verified"):
    raise RuntimeError("Ejecuta primero la comprobación de la celda 9.")
evaluation_output = RESULTS / ("validacion_" + uuid4().hex[:8])
evaluar_trigrama(evaluation_output)
evaluation = json.loads((evaluation_output / "metricas.json").read_text())
if (evaluation["optimizer_steps"] != 0 or evaluation["test_evaluated"]
        or evaluation["checkpoint_sha256"] != CHECKPOINT_SHA256):
    raise RuntimeError("La evaluación no corresponde al checkpoint y al protocolo seleccionados.")
for depth, values in evaluation["global"].items():
    print("Salida", depth, "· WER", f"{values['wer']:.2%}",
          f"({values['errores_palabras']}/{values['palabras']})", "· PER", f"{values['per']:.2%}",
          "· Frases", values["frases"], "· RTF", evaluation["timing"][depth]["rtf"])
print("Checkpoint SHA-256:", evaluation["checkpoint_sha256"])
print("Resultados:", evaluation_output)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: aaurre (aaurre-deusto). Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.18.7
wandb: Run data is saved locally in runs/tracking/wandb/run-20261101_203841-e3gy3ugk
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run speech_online_baseline_seed0_colab_trigrama_val
wandb: ⭐️ View project at https://wandb.ai/aaurre-deusto/tiiac-2026
wandb: 🚀 View run at https://wandb.ai/aaurre-deusto/tiiac-2026/runs/e3gy3ugk
wandb: - 0.004 MB of 0.004 MB uploaded
wandb: \ 0.004 MB of 0.004 MB uploaded
wandb:                                                                                
wandb: 
wandb: Run history:
wandb: trigrama/tiempo/capa_5/first_partial_input/observations ▁
wandb:       trigrama/tiempo/capa_5/first_partial_input/p50_ms ▁
wandb:       trigrama/tiempo/capa_5/first_partial_input/p95_ms ▁
wandb: tri